# ARENA 0.2 CNNs & ResNets - Complete Layman's ExplainerThis notebook walks through **every code cell** in the exercises notebook `0.2_CNNs_&_ResNets_exercises.ipynb`, explaining each line as if you've never seen code before.**How to read this notebook:** Each section mirrors a code cell from the exercises. First you'll see a plain-English explanation of what the code does and why, then the actual code with line-by-line comments.---

# Part 1: Setup Code## Cell 1 - Environment Setup (the "where am I?" cell)### What this cell does in plain EnglishThis cell figures out WHERE your code is running and makes sure all the necessary files are available. Think of it like arriving at a kitchen and checking:1. "Am I in my home kitchen or a restaurant kitchen?" (Colab vs local)2. "Do I have all my ingredients?" (are the exercise files downloaded?)3. "Let me set up my workstation" (point Python to the right folders)### Key concepts for absolute beginners**`import`** = "Go grab this toolbox so I can use it later."  Just like a chef says "bring me the knife set" before cooking, `import` loads a library (a collection of pre-written code) so you can use its tools.**`=`** (single equals) = "Store this value with this name."  It's NOT "equals" like in math. It's more like putting a label on a box. `x = 5` means "put 5 in a box labeled x."**Indentation (the spaces at the start of lines)** = "This code belongs inside that block above."  Python uses spacing to show structure. If a line is indented under an `if` statement, it only runs when that condition is true. Think of it like an outline:```If it's raining:    Bring an umbrella      <-- only happens if raining    Wear a raincoat        <-- only happens if rainingGo to work                 <-- happens regardless```

In [ ]:
# ---- LINE BY LINE BREAKDOWN ----import os          # Toolbox for interacting with your computer's file system                   # (checking if folders exist, getting current location, etc.)import sys         # Toolbox for interacting with Python itself                   # (where Python looks for files, what environment we're in)from pathlib import Path   # A nicer way to work with file paths                           # "from X import Y" means "from toolbox X, just grab tool Y"                           # Path turns strings like "/home/user/files" into objects                           # you can easily manipulate# ---- DETECTING THE ENVIRONMENT ----IN_COLAB = "google.colab" in sys.modules# sys.modules is a dictionary (lookup table) of all loaded Python modules# This line checks: "Is google.colab in that list?"# If yes -> we're running in Google Colab (IN_COLAB = True)# If no  -> we're running somewhere else (IN_COLAB = False)# This is like checking your GPS to know which kitchen you're in# ---- SETTING VARIABLES ----chapter = "chapter0_fundamentals"    # Just storing text labels for folder namesrepo = "ARENA_3.0"                   # These will be used later to build file pathsbranch = "main"                      # The Git branch to download from# ---- INSTALLING DEPENDENCIES ----# try/except is like saying: "Try this. If it fails, do this other thing instead."# # try:#     import torchinfo          # "Try to grab the torchinfo toolbox"# except:                       # "If that failed (it's not installed)..."#     %pip install torchinfo    # "...then install it"## %pip is a special Jupyter/Colab command to install Python packages# Think of it as going to the store to buy a tool you don't have yet

### The `root` variable - finding your way home```pythonroot = (    "/content"    if IN_COLAB    else "/root"    if repo not in os.getcwd()    else str(next(p for p in Path.cwd().parents if p.name == repo)))```This looks intimidating! Let's unpack it. This is a **ternary expression** (a compact if/else). Written out as normal English:```IF we're in Colab:    root = "/content"                    (Colab's default folder)ELSE IF we're NOT inside the ARENA repo:    root = "/root"                       (a generic starting point)ELSE (we ARE inside the ARENA repo):    root = walk UP the folder tree until you find the "ARENA_3.0" folder```**`os.getcwd()`** = "Get Current Working Directory" = "What folder am I in right now?"**`Path.cwd().parents`** = a list of all parent folders above you. If you're in `/home/user/ARENA_3.0/chapter0/exercises`, the parents are:- `/home/user/ARENA_3.0/chapter0`- `/home/user/ARENA_3.0`- `/home/user`- `/home`- `/`**`next(p for p in ... if p.name == repo)`** = "Walk through those parents and give me the FIRST one whose name is 'ARENA_3.0'"### The download block```pythonif Path(root).exists() and not Path(f"{root}/{chapter}").exists():```"If the root folder exists BUT the chapter folder doesn't exist inside it, we need to download it."The `!wget`, `!unzip`, `!mv`, `!rm` lines are **shell commands** (terminal commands, not Python). The `!` prefix tells Jupyter "run this as a terminal command, not as Python." They download and unpack the exercise files from GitHub.### The path setup```pythonif f"{root}/{chapter}/exercises" not in sys.path:    sys.path.append(f"{root}/{chapter}/exercises")os.chdir(f"{root}/{chapter}/exercises")```**`sys.path`** is Python's "search list" - when you `import something`, Python looks through these folders to find it. This line adds the exercises folder to that search list.**`os.chdir(...)`** = "Change Directory" = "Move to this folder." Like `cd` in a terminal. After this, all relative file paths are relative to the exercises folder.**`f"...{variable}..."`** is an **f-string** (formatted string). The `f` before the quotes means "replace anything in `{curly braces}` with its value." So `f"{root}/{chapter}"` might become `"/content/chapter0_fundamentals"`.

## Cell 2 - Importing Libraries (the "gather your tools" cell)This is the most important setup cell. Every line brings in a specific tool you'll need. Let's go through each one.

In [ ]:
# ---- STANDARD PYTHON LIBRARIES ----import json                        # For reading/writing JSON files (a data format)                                   # Used later to load ImageNet class labelsimport sys                         # System-level Python tools (we already saw this)from collections import namedtuple # A lightweight way to create simple data structures                                   # Like creating a form with named fields:                                   # Point = namedtuple("Point", ["x", "y"])                                   # p = Point(x=3, y=4)  -> p.x gives 3from dataclasses import dataclass  # A fancier way to create data structures (classes                                   # with automatic __init__). We'll see this later.from pathlib import Path           # Nice file path handling (seen above)# ---- THE BIG ONES: DEEP LEARNING LIBRARIES ----import einops                      # Einstein Operations - a library for reshaping tensors                                   # (multi-dimensional arrays) using readable string notation                                   # Instead of cryptic reshape commands, you write things like:                                   # "batch channels height width -> batch (channels height width)"import numpy as np                 # NumPy ("Numerical Python") - the foundation of scientific                                   # computing in Python. Provides fast array operations.                                   # "as np" means "I'll call it 'np' for short"                                   # So np.sqrt(4) means numpy's square root of 4import torch as t                  # PyTorch - THE deep learning framework. This is the big one.                                   # Everything in this notebook is built on PyTorch.                                   # "as t" means we'll write t.tensor() instead of torch.tensor()                                   # A TENSOR is just a multi-dimensional array of numbers:                                   #   - 0D tensor: a single number (scalar), e.g. 5                                   #   - 1D tensor: a list, e.g. [1, 2, 3]                                   #   - 2D tensor: a grid/table, e.g. [[1,2],[3,4]]                                   #   - 3D tensor: a cube of numbers                                   #   - 4D tensor: a batch of cubes (e.g. batch of images)import torch.nn as nn              # nn = "Neural Network" module                                   # Contains building blocks for neural networks:                                   # layers, activation functions, loss functions, etc.                                   # nn.Module is the base class EVERYTHING inherits fromimport torch.nn.functional as F    # The FUNCTIONAL versions of nn operations                                   # nn.ReLU() creates an OBJECT you can reuse                                   # F.relu(x) is a one-shot FUNCTION call                                   # F.cross_entropy() is the loss function we'll use for trainingimport torchinfo                   # Prints nice summaries of model architectures                                   # (how many parameters, what shapes flow through, etc.)# ---- DISPLAY AND TYPE TOOLS ----from IPython.display import display  # For showing images in Jupyter notebooksfrom jaxtyping import Float, Int   # Type hints that describe tensor SHAPES                                   # Float[Tensor, "batch channels height width"]                                   # means "a Tensor of floats with these 4 dimensions"                                   # These don't enforce anything - they're documentationfrom PIL import Image              # PIL = Python Imaging Library                                   # For loading and manipulating image files (.jpg, .png)from rich import print as rprint   # A fancier print function with colors and formattingfrom rich.table import Table       # For printing nice tables in the terminal# ---- PYTORCH DATA TOOLS ----from torch import Tensor           # Import the Tensor class directly so we can use it                                   # in type hints without writing t.Tensor every timefrom torch.utils.data import DataLoader, Subset# DataLoader: feeds data to your model in BATCHES#   Think of it as a waiter bringing plates of food (batches of images)#   to the kitchen (model) one at a time, rather than dumping everything at once# Subset: takes a slice of a dataset (e.g., first 10,000 images out of 60,000)from torchvision import datasets, models, transforms# datasets: pre-packaged datasets like MNIST (handwritten digits)# models: pre-built architectures like ResNet (we'll load a pre-trained one)# transforms: functions to preprocess images before feeding to modelsfrom tqdm.notebook import tqdm     # Progress bars! Shows you how far along a loop is.                                   # tqdm wraps around any iterable:                                   # for x in tqdm(my_list): ...                                   # displays: 45%|████████████         | 45/100# ---- LOCAL PROJECT IMPORTS ----# These set up paths so Python can find the test files and utilitieschapter = "chapter0_fundamentals"section = "part2_cnns"root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())exercises_dir = root_dir / chapter / "exercises"section_dir = exercises_dir / sectionif str(exercises_dir) not in sys.path:    sys.path.append(str(exercises_dir))import part2_cnns.tests as tests   # The test suite - functions that check if your                                   # implementations are correctimport part2_cnns.utils as utils   # Utility/helper functions provided by ARENAfrom plotly_utils import line      # A wrapper around Plotly for making line charts                                   # (used to visualize training loss and accuracy)

### What's the difference between `import X` and `from X import Y`?| Syntax | What it does | How you use it ||---|---|---|| `import torch` | Loads the entire `torch` library | `torch.tensor([1,2,3])` || `import torch as t` | Same, but nickname it `t` | `t.tensor([1,2,3])` || `from torch import Tensor` | Loads JUST `Tensor` from torch | `Tensor([1,2,3])` || `from torch.utils.data import DataLoader` | Loads one specific tool from deep inside torch | `DataLoader(dataset)` |Think of it like a toolbox:- `import torch` = "Bring me the entire PyTorch toolbox"- `from torch import Tensor` = "Just bring me the wrench from the PyTorch toolbox"- `import torch as t` = "Bring the toolbox but let me call it 't' for short"### What is `as`?`as` creates a **nickname** (alias). `import numpy as np` means "load numpy but I'll call it np." This is purely for convenience - typing `np.sqrt(4)` is faster than `numpy.sqrt(4)`. These nicknames are conventions that everyone uses, so when you see `t.` in code, experienced Python users know it means PyTorch.

---# Part 2: Making Your Own Modules## What is `nn.Module`?Everything in PyTorch neural networks inherits from `nn.Module`. Think of it as a **blueprint for a machine part**. Every part (layer) in a neural network:1. Has an `__init__` method where you set it up (buy the parts)2. Has a `forward` method that describes what it DOES when data flows through itWhen you stack modules together, data flows through them like water through pipes: in one end, transformed, out the other.## `class` - What is it?A **class** is a blueprint for creating objects. Think of it like a cookie cutter:```pythonclass Dog:                    # The blueprint    def __init__(self, name): # How to set up a new dog        self.name = name      # Each dog remembers its name        def bark(self):           # Something every dog can do        return f"{self.name} says Woof!"my_dog = Dog("Rex")           # Create an actual dog from the blueprintmy_dog.bark()                 # "Rex says Woof!"```**`self`** = "the specific object we're talking about." When Rex barks, `self` is Rex. When Spot barks, `self` is Spot. It's how the object refers to itself.**`def`** = "define a function" (a reusable block of code)**`__init__`** (with double underscores, called "dunder init") = the **constructor**. It runs automatically when you create a new instance. `Dog("Rex")` automatically calls `__init__` with name="Rex".## Cell 3 - ReLU (your first module)**ReLU** stands for "Rectified Linear Unit." Despite the fancy name, it does one incredibly simple thing:- If a number is positive, keep it- If a number is negative, make it zeroThat's it. `ReLU(5) = 5`, `ReLU(-3) = 0`, `ReLU(0) = 0`.**Why does this matter?** Without ReLU (or similar functions), stacking linear layers would just collapse into one linear layer. ReLU introduces **nonlinearity**, which lets the network learn curves and complex patterns instead of just straight lines.

In [ ]:
class ReLU(nn.Module):    # "class ReLU(nn.Module):" means:    # "Create a new blueprint called ReLU that INHERITS from nn.Module"    #     # Inheriting means ReLU gets all of nn.Module's built-in abilities for free    # (like being able to track parameters, move to GPU, etc.)    # It's like saying "a ReLU IS A type of Module" - just a specialized one.        # Notice: there's NO __init__ method here!    # That's because ReLU has no settings to configure, no weights to store.    # It just does one fixed thing. The parent class (nn.Module) has its own    # __init__ that handles the basics, and that's enough.        def forward(self, x: Tensor) -> Tensor:        # "def" = define a function (a reusable recipe)        # "forward" = the special method nn.Module calls when data passes through        # "self" = this specific ReLU instance        # "x: Tensor" = the input, labeled as a Tensor (type hint, not enforced)        # "-> Tensor" = this function returns a Tensor (also just a hint)                return t.maximum(x, t.tensor(0.0))        # t.maximum(a, b) compares two tensors element-by-element and keeps the bigger one        # t.tensor(0.0) creates a tensor containing just the number 0        #        # So for input [-3, 5, -1, 8, 0]:        #   compare each to 0:        #   max(-3, 0) = 0        #   max(5, 0)  = 5        #   max(-1, 0) = 0        #   max(8, 0)  = 8        #   max(0, 0)  = 0        #   Result: [0, 5, 0, 8, 0]# This line TESTS your implementation:# tests.test_relu(ReLU)# It creates a ReLU instance, passes in known inputs, and checks if outputs are correct

## Cell 4 - Linear Layer (the workhorse of neural networks)A Linear layer does matrix multiplication: it takes a list of numbers and computes weighted sums to produce a new list of numbers. Every neuron in the output looks at ALL the inputs, but with different weights.**Real-world analogy:** Imagine you're a judge scoring ice skaters on 5 criteria (speed, grace, jumps, spins, artistry). Each judge has different WEIGHTS for how important each criterion is. One judge might care more about jumps, another about artistry. Each judge produces ONE score by combining all 5 inputs with their personal weights. If you have 3 judges, you get 3 scores from 5 inputs. That's a Linear(5, 3) layer.### What's `nn.Parameter`?A **Parameter** is a tensor that PyTorch knows it needs to UPDATE during training. Regular tensors are just data. Parameters are the "knobs" the network turns to learn.### What's `super().__init__()`?When your class inherits from another class (like ReLU inherits from nn.Module), you need to tell the parent class to set itself up too. `super()` means "my parent class" and `.__init__()` means "run your setup." It's like saying "before I customize my house, let the builders finish the foundation first."

In [ ]:
class Linear(nn.Module):    def __init__(self, in_features: int, out_features: int, bias=True):        # in_features: how many numbers come IN (e.g., 784 for a 28x28 image)        # out_features: how many numbers come OUT (e.g., 100 neurons)        # bias: whether to add a constant offset (default: yes)                super().__init__()        # "Hey parent class (nn.Module), do your setup first!"        # This is REQUIRED. Without it, PyTorch can't track this module's parameters.                self.in_features = in_features        self.out_features = out_features        # Store these values so we can use them later (e.g., in extra_repr)        # "self.X = X" means "save X as a property of this object"                # ---- WEIGHT INITIALIZATION ----        sf = 1 / np.sqrt(in_features)        # sf = "scaling factor"        # np.sqrt() = NumPy's square root function        # If in_features=784, then sf = 1/sqrt(784) = 1/28 ≈ 0.036        #        # WHY? If weights are too big, numbers EXPLODE as they pass through layers        # If too small, numbers VANISH to zero. This scaling keeps things stable.        # It's called "Kaiming initialization" and the math behind it ensures        # that the variance of outputs ≈ variance of inputs.                weight = sf * (2 * t.rand(out_features, in_features) - 1)        # Step by step:        # 1. t.rand(out_features, in_features) -> random numbers in [0, 1)        #    Creates a GRID of random numbers. If out=100, in=784, that's a        #    100-row x 784-column table of random values between 0 and 1        #        # 2. (2 * ... - 1) -> shifts range to [-1, 1)        #    Multiply by 2: now [0, 2). Subtract 1: now [-1, 1).        #    We want weights centered around zero, not biased positive.        #        # 3. sf * ... -> scale down by 1/sqrt(in_features)        #    Final range: roughly [-0.036, 0.036] for in_features=784                self.weight = nn.Parameter(weight)        # Wrap the tensor in nn.Parameter to tell PyTorch:        # "This is a LEARNABLE weight. Include it when I ask for model.parameters()        #  and update it during training."                if bias:            bias = sf * (2 * t.rand(out_features) - 1)            # Same random initialization, but just a 1D list (one bias per output)            self.bias = nn.Parameter(bias)        else:            self.bias = None            # No bias? Set it to None (Python's word for "nothing")        def forward(self, x: Tensor) -> Tensor:        # x: shape (*, in_features) - the * means "any number of batch dimensions"        # For example: (64, 784) means 64 images, each with 784 pixel values                x = einops.einsum(x, self.weight, "... in_feats, out_feats in_feats -> ... out_feats")        # EINSUM = "Einstein Summation" - a compact way to express tensor operations        #        # The string describes what happens:        #   "... in_feats"           = input x has some batch dims (...) and in_features        #   "out_feats in_feats"     = weight matrix is (out_features x in_features)        #   "-> ... out_feats"       = output keeps batch dims, replaces in_feats with out_feats        #        # What actually happens: for each output neuron, compute the DOT PRODUCT        # of the input with that neuron's row of weights. This is matrix multiplication.        #        # Concretely for one image with 784 pixels and 100 output neurons:        #   output[0] = input[0]*weight[0,0] + input[1]*weight[0,1] + ... + input[783]*weight[0,783]        #   output[1] = input[0]*weight[1,0] + input[1]*weight[1,1] + ... + input[783]*weight[1,783]        #   ... (100 such sums)                if self.bias is not None:            x += self.bias            # "+=" means "add to itself"            # x += self.bias is short for x = x + self.bias            # Adds one bias value to each output neuron            # Broadcasting: bias shape (100,) automatically applies to every image in the batch                return x        def extra_repr(self) -> str:        # This controls what you see when you print(model)        # e.g., "Linear(in_features=784, out_features=100, bias=True)"        return f"in_features={self.in_features}, out_features={self.out_features}, bias={self.bias is not None}"        # "self.bias is not None" returns True if bias exists, False if it's None

## Cell 5 - Flatten (the shape adapter)**The problem:** Convolutional layers output 3D data (channels x height x width). Linear layers expect 1D data (a flat list of numbers). Flatten bridges this gap.**Analogy:** Imagine you have a filing cabinet with 32 drawers, each containing an 8x8 grid of cards. Flatten dumps all cards from all drawers into one big pile. 32 x 8 x 8 = 2,048 cards in a flat stack.By default, Flatten keeps the batch dimension (dimension 0) intact and flattens everything else. So a batch of 64 images, each with shape (32, 8, 8), becomes 64 flat vectors of length 2048.

In [ ]:
class Flatten(nn.Module):    def __init__(self, start_dim: int = 1, end_dim: int = -1) -> None:        super().__init__()        self.start_dim = start_dim   # Which dimension to START flattening from                                     # Default: 1 (skip dim 0, which is the batch)        self.end_dim = end_dim       # Which dimension to STOP flattening at                                     # Default: -1 (the last dimension)                                     # -1 is Python shorthand for "the last one"        def forward(self, input: Tensor) -> Tensor:        shape = input.shape        # .shape returns the dimensions as a tuple        # e.g., for a tensor of shape (64, 32, 8, 8):        #   shape = (64, 32, 8, 8)        #   shape[0] = 64 (batch size)        #   shape[1] = 32 (channels)        #   shape[2] = 8  (height)        #   shape[3] = 8  (width)                # Handle negative indexing: -1 means "last dimension"        start_dim = self.start_dim        end_dim = self.end_dim if self.end_dim >= 0 else len(shape) + self.end_dim        # len(shape) = number of dimensions (e.g., 4)        # If end_dim = -1: end_dim = 4 + (-1) = 3 (index of last dim)                # Split the shape into three parts:        shape_left = shape[:start_dim]        # shape[:1] = (64,) -- everything BEFORE the flattening range        # The : is "slice" notation: shape[:1] means "from start up to (not including) index 1"                shape_right = shape[end_dim + 1 :]        # shape[4:] = () -- everything AFTER the flattening range (nothing in this case)                shape_middle = t.prod(t.tensor(shape[start_dim : end_dim + 1])).item()        # shape[1:4] = (32, 8, 8) -- the dimensions we're flattening        # t.tensor(...) converts to a tensor        # t.prod(...) multiplies all elements: 32 * 8 * 8 = 2048        # .item() converts a single-element tensor to a plain Python number                return t.reshape(input, shape_left + (shape_middle,) + shape_right)        # Combine: (64,) + (2048,) + () = (64, 2048)        # t.reshape changes the shape without changing the data        # The numbers are the same, just organized differently        #        # Note: (2048,) with a comma is a TUPLE with one element        # In Python, (2048) without comma is just the number 2048 in parentheses        # (2048,) with comma is a tuple containing 2048        # This matters because + concatenates tuples: (64,) + (2048,) = (64, 2048)

## Cell 6 - SimpleMLP (your first neural network!)This puts together everything above into a complete digit recognizer:```Input (28x28 image)    |    vFlatten (784 numbers)    |    vLinear (784 -> 100)    <-- "look at all pixels, produce 100 pattern signals"    |    vReLU                   <-- "zero out negative signals" (introduces nonlinearity)    |    vLinear (100 -> 10)     <-- "combine patterns into 10 digit scores"    |    vOutput (10 logits: one score per digit 0-9)```**Why 28*28 = 784?** MNIST images are 28 pixels wide by 28 pixels tall.  **Why 100?** Arbitrary choice - it's a "hidden layer" size. Could be 50, 200, etc.  **Why 10?** There are 10 possible digits (0 through 9).

In [ ]:
class SimpleMLP(nn.Module):    def __init__(self):        super().__init__()        # Set up the four "machine parts" of our network        self.flatten = Flatten()                              # Shape adapter: 28x28 -> 784        self.linear1 = Linear(in_features=28 * 28, out_features=100)  # 784 inputs -> 100 outputs        self.relu = ReLU()                                    # Zero out negatives        self.linear2 = Linear(in_features=100, out_features=10)       # 100 inputs -> 10 outputs                # By saving each as self.something, PyTorch auto-discovers them        # and knows to include their Parameters when training        def forward(self, x: Tensor) -> Tensor:        return self.linear2(self.relu(self.linear1(self.flatten(x))))        # Read INSIDE-OUT (like math: f(g(x)) means "first apply g, then f"):        #        # Step 1: self.flatten(x)          Input (64, 1, 28, 28) -> (64, 784)        # Step 2: self.linear1(...)        (64, 784) -> (64, 100)        # Step 3: self.relu(...)           (64, 100) -> (64, 100)  [negatives become 0]        # Step 4: self.linear2(...)        (64, 100) -> (64, 10)        #        # The 64 is the batch size (64 images processed simultaneously)        # Each image independently flows through the same pipeline        #        # This nesting style is common but can be hard to read. An equivalent way:        #   x = self.flatten(x)        #   x = self.linear1(x)        #   x = self.relu(x)        #   x = self.linear2(x)        #   return x

---# Part 3: Training Neural Networks## Transforms, Datasets & DataLoadersBefore a neural network can look at images, the images need to be converted into tensors (numbers) and preprocessed. This section sets up that pipeline.**Analogy:** Before a chef can cook ingredients, they need to be washed, peeled, and chopped into uniform sizes. Transforms are the "food prep" step for data.

In [ ]:
# ---- TRANSFORMS: the "food prep" pipeline ----MNIST_TRANSFORM = transforms.Compose(    [        transforms.ToTensor(),        transforms.Normalize(0.1307, 0.3081),    ])# transforms.Compose([...]) chains multiple transforms together into a PIPELINE# It's like saying "do step 1, then step 2, then step 3" in order## transforms.ToTensor():#   Converts a PIL Image (a picture file) into a PyTorch Tensor#   Also scales pixel values from 0-255 (standard image range) to 0.0-1.0#   A 28x28 grayscale image becomes a tensor of shape (1, 28, 28)#   The "1" is the channel dimension (1 channel for grayscale, 3 for RGB color)## transforms.Normalize(0.1307, 0.3081):#   Applies the formula: pixel = (pixel - mean) / std_deviation#   0.1307 is the average pixel value across ALL MNIST images#   0.3081 is the standard deviation of pixel values across ALL MNIST images#   After this, the data is centered around 0 with roughly unit variance#   WHY? Neural networks learn better when inputs are centered around zero#   and have similar scales. It's like converting all measurements to the#   same unit system before comparing them.

## The `get_mnist` function - loading the dataset

In [ ]:
def get_mnist(trainset_size: int = 10_000, testset_size: int = 1_000) -> tuple[Subset, Subset]:    # This function RETURNS two things: a training set and a test set    #     # "-> tuple[Subset, Subset]" is a type hint saying "this returns a pair of Subsets"    # A tuple is an ordered, immutable collection: (thing1, thing2)    #    # 10_000 and 1_000: the underscores are just for readability (like commas in 10,000)    # Python ignores them: 10_000 == 10000        # ---- DOWNLOADING THE DATA ----    mnist_trainset = datasets.MNIST(        exercises_dir / "data",    # WHERE to save/find the downloaded files        train=True,                # Get the TRAINING portion (60,000 images)        download=True,             # Download if not already on disk        transform=MNIST_TRANSFORM  # Apply our Compose pipeline to each image    )    mnist_testset = datasets.MNIST(        exercises_dir / "data",        train=False,               # Get the TEST portion (10,000 images)        download=True,        transform=MNIST_TRANSFORM    )        # ---- TAKING A SUBSET ----    mnist_trainset = Subset(mnist_trainset, indices=range(trainset_size))    mnist_testset = Subset(mnist_testset, indices=range(testset_size))    # Full MNIST has 60,000 train + 10,000 test images    # For speed, we only use the first 10,000 train + 1,000 test    # range(10_000) produces [0, 1, 2, ..., 9999]    # Subset(dataset, indices) picks only those items        return mnist_trainset, mnist_testset    # Returns BOTH datasets as a tuple    # The caller can "unpack" them: train, test = get_mnist()# ---- CREATING DATALOADERS ----mnist_trainset, mnist_testset = get_mnist()# "Unpacking": the two returned values get assigned to two variablesmnist_trainloader = DataLoader(mnist_trainset, batch_size=64, shuffle=True)# DataLoader wraps a dataset and serves it in BATCHES## batch_size=64: serve 64 images at a time#   WHY batches? Two reasons:#   1. Memory: can't fit all 10,000 images in GPU memory at once#   2. Learning: averaging gradients over a batch gives smoother updates## shuffle=True: randomize the order each epoch#   WHY shuffle? Without it, the model might learn the ORDER of images#   rather than the CONTENT. Shuffling prevents this pattern-memorization.mnist_testloader = DataLoader(mnist_testset, batch_size=64, shuffle=False)# shuffle=False for test data: we're just scoring, order doesn't matter# ---- INSPECTING THE DATA ----for img_batch, label_batch in mnist_testloader:    print(f"{img_batch.shape=}")    # Shows: torch.Size([64, 1, 28, 28])    print(f"{label_batch.shape=}")  # Shows: torch.Size([64])    break                           # "break" exits the loop after first iteration                                    # We just wanted to peek at one batch# img_batch.shape = (64, 1, 28, 28) means:#   64 images, each with 1 channel (grayscale), 28 pixels tall, 28 pixels wide## label_batch.shape = (64,) means:#   64 labels, each is an integer 0-9 (which digit it is)## f"{img_batch.shape=}" is a Python trick: the = inside the f-string# prints BOTH the variable name AND its value: "img_batch.shape=torch.Size([64, 1, 28, 28])"for img, label in mnist_testset:    print(f"{img.shape=}")          # Shows: torch.Size([1, 28, 28]) -- single image    print(f"{label=}")              # Shows: label=7 (or whatever digit)    break

## Device Selection (CPU vs GPU)Neural network training involves billions of simple math operations (multiply, add). GPUs (Graphics Processing Units) can do thousands of these operations simultaneously, making training 10-100x faster than CPU.**Analogy:** CPU is like one very smart mathematician. GPU is like 1,000 average mathematicians working in parallel. For tasks like "multiply these 10,000 pairs of numbers," the parallel army wins by a landslide.

In [ ]:
device = t.device(    "mps" if t.backends.mps.is_available()      # Apple Silicon GPU (M1/M2/M3 Macs)    else "cuda" if t.cuda.is_available()          # NVIDIA GPU    else "cpu"                                    # Fallback: regular processor)# t.device() creates a "device" object that tells PyTorch where to put tensors# This ternary chain checks GPUs in order of preference:#   1. MPS (Apple GPU) - if available, use it#   2. CUDA (NVIDIA GPU) - if available, use it (most common for ML)#   3. CPU - always available, but slower## On Google Colab with GPU enabled, this will say "cuda"# If you forgot to enable GPU in Colab: Runtime > Change runtime type > GPUprint(device)# Will print one of: "mps", "cuda", or "cpu"# If it says "cpu" and you're on Colab, enable GPU in runtime settings!

## The Training Loop (Cell 39) - Teaching the NetworkThis is where the actual LEARNING happens. Let's understand the big picture first:### The Training Recipe1. Show the model a batch of images2. The model makes predictions (probably wrong at first)3. Measure HOW wrong the predictions are (loss)4. Figure out which direction to adjust each weight to reduce the wrongness (backpropagation)5. Nudge the weights slightly in that direction (optimizer step)6. Repeat thousands of times**Analogy:** Imagine learning to throw darts blindfolded. After each throw, someone tells you "you were 3 inches to the left and 2 inches high." You adjust. After thousands of throws, you get pretty good. The "3 inches left, 2 inches high" is the **loss**. The adjustment is the **optimizer step**.

In [ ]:
model = SimpleMLP().to(device)# 1. SimpleMLP() creates a new model with RANDOM weights (it knows nothing yet)# 2. .to(device) moves it to the GPU (or CPU). The model and data must be on#    the same device, or PyTorch will error.batch_size = 128epochs = 3# An EPOCH is one complete pass through the entire training set# 3 epochs = the model sees every training image 3 times# (Each time, the weights have been updated, so it learns new things)mnist_trainset, _ = get_mnist()# The _ is Python convention for "I don't care about this value"# get_mnist() returns (trainset, testset), but we only need trainset heremnist_trainloader = DataLoader(mnist_trainset, batch_size=batch_size, shuffle=True)# Serve 128 images at a time, in random orderoptimizer = t.optim.Adam(model.parameters(), lr=1e-3)# The OPTIMIZER is the algorithm that updates weights based on gradients## model.parameters() returns ALL learnable weights in the model#   (the weight matrices and bias vectors from both Linear layers)## Adam is a popular optimizer. Think of it as a "smart weight adjuster"#   - Basic optimizer (SGD): always takes the same size step#   - Adam: adapts step size per parameter, has momentum (remembers past gradients)## lr=1e-3 = learning rate = 0.001#   This controls HOW BIG each adjustment is#   Too big (0.1): overshoots, can't settle on good weights#   Too small (0.000001): learns painfully slowly#   0.001 is a common "safe default"##   1e-3 is scientific notation: 1 x 10^(-3) = 0.001loss_list = []# An empty list to record the loss at each step (for plotting later)# ---- THE MAIN TRAINING LOOP ----for epoch in range(epochs):    # range(3) produces [0, 1, 2], so this loop runs 3 times        pbar = tqdm(mnist_trainloader)    # Wrap the dataloader in tqdm to get a progress bar    # Without tqdm, you'd just write: for imgs, labels in mnist_trainloader:        for imgs, labels in pbar:        # Each iteration, the DataLoader gives us ONE BATCH:        # imgs: shape (128, 1, 28, 28) - 128 images        # labels: shape (128,) - 128 correct answers (digits 0-9)                # ---- STEP 1: MOVE DATA TO GPU ----        imgs, labels = imgs.to(device), labels.to(device)        # Data starts on CPU. Model is on GPU. They must match.        # .to(device) creates a copy on the target device                # ---- STEP 2: FORWARD PASS (make predictions) ----        logits = model(imgs)        # model(imgs) calls model.forward(imgs) behind the scenes        # logits: shape (128, 10) - for each of 128 images, 10 scores (one per digit)        # These are RAW scores, not probabilities        # Higher score = model thinks this digit is more likely                # ---- STEP 3: COMPUTE LOSS (how wrong are we?) ----        loss = F.cross_entropy(logits, labels)        # Cross entropy loss measures: "how surprised would you be if the true        # answer was X, given the probabilities you predicted?"        #        # It does three things internally:        #   1. Apply softmax to logits -> convert to probabilities (sum to 1)        #   2. Look up the probability assigned to the CORRECT class        #   3. Take -log of that probability        #        # If model was 99% confident in the right answer: loss ≈ 0.01 (low, good!)        # If model was 1% confident in the right answer: loss ≈ 4.6 (high, bad!)        # Loss is a single number summarizing "how wrong" across the whole batch                # ---- STEP 4: BACKWARD PASS (compute gradients) ----        loss.backward()        # This is the MAGIC of PyTorch. It automatically computes:        # "For each weight in the model, how much would the loss change        #  if I nudged that weight slightly?"        # These "how much would loss change" values are called GRADIENTS        # They're stored inside each parameter as param.grad        #        # This uses the chain rule from calculus, applied automatically        # through every operation that produced the loss                # ---- STEP 5: UPDATE WEIGHTS ----        optimizer.step()        # The optimizer looks at each parameter's gradient and UPDATES the weight:        #   weight = weight - learning_rate * gradient        # (Adam is more complex than this, but that's the basic idea)        # If the gradient says "increasing this weight increases loss,"        # then we DECREASE the weight (to reduce loss)                # ---- STEP 6: ZERO GRADIENTS ----        optimizer.zero_grad()        # CRITICAL: PyTorch ACCUMULATES gradients by default        # Without zeroing, gradients from batch 2 would ADD to gradients from batch 1        # We want fresh gradients for each batch, so we zero them out        # This is a common source of bugs if forgotten!                # ---- LOGGING ----        loss_list.append(loss.item())        # loss.item() converts a single-element tensor to a plain Python float        # We save it to plot later                pbar.set_postfix(epoch=f"{epoch + 1}/{epochs}", loss=f"{loss:.3f}")        # Updates the progress bar to show current epoch and loss        # f"{loss:.3f}" formats the loss to 3 decimal places

## Plotting the Loss

In [ ]:
# line() is a helper function from plotly_utils that creates a line chart# # line(#     loss_list,                          # The data to plot (y-axis values)#     x_max=epochs * len(mnist_trainset), # Scale x-axis to show "examples seen"#     labels={"x": "Examples seen", "y": "Cross entropy loss"},  # Axis labels#     title="SimpleMLP training on MNIST",#     width=700,# )## You should see the loss dropping from ~2.3 (random guessing among 10 classes:# -log(1/10) ≈ 2.3) down to ~0.3 or lower.## The line will be jagged (noisy) because each point is the loss on one batch,# and batches vary in difficulty. The TREND should be clearly downward.

## The `@dataclass` pattern - Organizing SettingsAs code gets more complex, you end up with many settings (batch size, learning rate, epochs, etc.). A **dataclass** is a clean way to bundle them together instead of passing 10 separate arguments to a function.**Analogy:** Instead of writing a cooking recipe that says "use flour=2cups, sugar=1cup, eggs=3, butter=0.5cup..." you create a Recipe Card with all the defaults listed, and you can override any of them.

In [ ]:
@dataclassclass SimpleMLPTrainingArgs:    # @dataclass is a DECORATOR (the @ symbol)    # A decorator modifies the class that follows it    # @dataclass automatically creates an __init__ method from the fields below    # So you DON'T need to write:    #   def __init__(self, batch_size=64, epochs=3, learning_rate=1e-3):    #       self.batch_size = batch_size    #       self.epochs = epochs    #       self.learning_rate = learning_rate    # The @dataclass does all that for you!        batch_size: int = 64           # How many images per batch    epochs: int = 3                # How many full passes through the data    learning_rate: float = 1e-3    # How big the weight adjustments are        # "int" and "float" are TYPE HINTS:    #   int = whole numbers (1, 2, 64, 128)    #   float = decimal numbers (0.001, 3.14, 1e-3)    #   These are documentation only - Python doesn't enforce them        # The = values are DEFAULTS. You can override when creating:    #   args = SimpleMLPTrainingArgs()                    # uses all defaults    #   args = SimpleMLPTrainingArgs(batch_size=128)      # overrides batch_size only    #   args = SimpleMLPTrainingArgs(epochs=10, lr=0.01)  # overrides two

## The Refactored Training FunctionThe training loop from before is now wrapped in a **function**. This makes it reusable — you can call `train()` with different settings without copy-pasting the whole loop.

In [ ]:
def train(args: SimpleMLPTrainingArgs) -> tuple[list[float], SimpleMLP]:    # "def" = define a function    # "args: SimpleMLPTrainingArgs" = this function takes one argument, a dataclass    # "-> tuple[list[float], SimpleMLP]" = it returns a (list of losses, trained model)        model = SimpleMLP().to(device)        mnist_trainset, _ = get_mnist()    mnist_trainloader = DataLoader(mnist_trainset, batch_size=args.batch_size, shuffle=True)    #                                                       ^^^^^^^^^^^^^^^^    # Notice: instead of hardcoded "128", we use args.batch_size    # This is the benefit of the dataclass - all settings come from one place        optimizer = t.optim.Adam(model.parameters(), lr=args.learning_rate)    #                                                ^^^^^^^^^^^^^^^^^    # Same here: learning rate comes from args        loss_list = []        for epoch in range(args.epochs):        pbar = tqdm(mnist_trainloader)        for imgs, labels in pbar:            imgs, labels = imgs.to(device), labels.to(device)            logits = model(imgs)            loss = F.cross_entropy(logits, labels)            loss.backward()            optimizer.step()            optimizer.zero_grad()            loss_list.append(loss.item())            pbar.set_postfix(epoch=f"{epoch + 1}/{args.epochs}", loss=f"{loss:.3f}")        return loss_list, model    # Returns BOTH the loss history (for plotting) and the trained modelargs = SimpleMLPTrainingArgs()   # Create settings with defaultsloss_list, model = train(args)   # Train! Unpack the two returned values

## The Validation Loop - Checking Homework on New ProblemsThe training loop teaches the model. The validation loop TESTS it on images it has **never seen during training**. This tells you if the model actually learned to recognize digits, or just memorized the training images.### Key differences from training:| | Training | Validation ||---|---|---|| **Purpose** | Learn (update weights) | Test (measure performance) || **Data** | Training set | Test set (never seen before) || **Metric** | Loss (how wrong) | Accuracy (% correct) || **Gradients** | Yes (needed for learning) | No (just observing) || **Updates weights?** | Yes | No |

In [ ]:
def train(args: SimpleMLPTrainingArgs) -> tuple[list[float], list[float], SimpleMLP]:    # Note: now returns THREE things: loss_list, accuracy_list, model        model = SimpleMLP().to(device)        mnist_trainset, mnist_testset = get_mnist()    # ^^^^^^^^^^^^^^^^^^^^^^^^^^    # CHANGED: We now keep BOTH datasets (previously threw away testset with _)        mnist_trainloader = DataLoader(mnist_trainset, batch_size=args.batch_size, shuffle=True)    mnist_testloader = DataLoader(mnist_testset, batch_size=args.batch_size, shuffle=False)    # NEW: A dataloader for the test set too    # shuffle=False because order doesn't matter when just scoring        optimizer = t.optim.Adam(model.parameters(), lr=args.learning_rate)        loss_list = []    accuracy_list = []     # NEW: track accuracy at the end of each epoch        for epoch in range(args.epochs):        # ========== TRAINING PHASE (same as before) ==========        pbar = tqdm(mnist_trainloader)        for imgs, labels in pbar:            imgs, labels = imgs.to(device), labels.to(device)            logits = model(imgs)            loss = F.cross_entropy(logits, labels)            loss.backward()            optimizer.step()            optimizer.zero_grad()            loss_list.append(loss.item())            pbar.set_postfix(epoch=f"{epoch + 1}/{args.epochs}", loss=f"{loss:.3f}")                # ========== VALIDATION PHASE (all new!) ==========        num_correct_classifications = 0        # Start counting correct predictions from zero                for imgs, labels in mnist_testloader:            # Loop through ALL test batches (no progress bar needed - it's fast)                        imgs, labels = imgs.to(device), labels.to(device)            # Move to same device as model                        with t.inference_mode():                logits = model(imgs)            # "with" creates a CONTEXT MANAGER - a temporary environment            # t.inference_mode() tells PyTorch:            #   "I'm just LOOKING, not LEARNING. Don't waste memory tracking            #    which operations happened (no gradient computation needed)."            # This makes the forward pass faster and uses less memory.            # It's like switching from "recording mode" to "playback mode"                        predictions = t.argmax(logits, dim=1)            # argmax = "argument of the maximum" = INDEX of the highest value            #             # logits shape: (64, 10) - for each image, 10 scores            # dim=1 means "find the max along dimension 1 (the 10 classes)"            #             # Example for one image:            #   logits = [-2.1, -0.5, 8.3, 1.2, -1.0, 0.3, 0.1, -0.8, 0.5, -1.4]            #                          ^^^ highest at index 2            #   argmax -> 2  (model predicts digit "2")            #            # For a batch of 64: produces 64 predictions                        num_correct_classifications += (predictions == labels).sum().item()            # Step by step:            # 1. (predictions == labels) compares element-by-element            #    Result: tensor of True/False, e.g., [True, False, True, True, ...]            #    True where prediction matches the actual label            #            # 2. .sum() counts the Trues (True=1, False=0)            #    e.g., if 50 out of 64 are correct: sum = 50            #            # 3. .item() converts tensor(50) to plain Python int 50            #            # 4. += adds to running total            #    "+=" is shorthand: x += 5 means x = x + 5                accuracy = num_correct_classifications / len(mnist_testset)        # Total correct / total test images = fraction correct        # e.g., 950 / 1000 = 0.95 = 95% accuracy                accuracy_list.append(accuracy)        # Log once per epoch (not per batch, since model doesn't change during validation)        return loss_list, accuracy_list, model

## Plotting Loss and Accuracy Together

In [ ]:
# line(#     y=[loss_list, [0.1] + accuracy_list],#     use_secondary_yaxis=True,#     x_max=args.epochs * len(mnist_trainset),#     labels={"x": "Num examples seen", "y1": "Cross entropy loss", "y2": "Test Accuracy"},#     title="SimpleMLP training on MNIST",#     width=800,# )## This plots TWO lines on the same chart:#   y1 (left axis): loss_list - the training loss (should go DOWN)#   y2 (right axis): accuracy - the test accuracy (should go UP)## [0.1] + accuracy_list prepends 0.1 (10%) as the starting accuracy# WHY 10%? With 10 digit classes, random guessing gets 1/10 = 10% right## use_secondary_yaxis=True puts the two metrics on different y-axes# because loss (~0-2.3) and accuracy (~0-1) have very different scales## You should see:#   - Loss dropping from ~2.3 to ~0.3#   - Accuracy climbing from 10% to ~95-97%

---# Part 4: Convolutions## What are Convolutions?So far, the SimpleMLP treated each image as a flat list of 784 numbers, ignoring the spatial structure entirely. A pixel in the top-left corner was treated no differently than its neighbor. **Convolutions** fix this by processing images in a way that respects spatial relationships.**Analogy:** Imagine you're looking at a photo through a small magnifying glass. You can only see a small patch at a time (say 3x3 pixels). You slide the magnifying glass across the entire photo, and at each position, you compute a single number that summarizes what you see in that patch. The "magnifying glass" is called a **kernel** (or filter), and the numbers it computes form a new, smaller image.### How it works:1. Take a small grid of weights (e.g., 3x3) — this is the **kernel**2. Place it at the top-left corner of the image3. Multiply each kernel weight by the pixel underneath it4. Sum all those products into one number5. Slide the kernel one position to the right and repeat6. When you hit the right edge, move down one row and start again7. The grid of output numbers is called a **feature map**### Key parameters:- **in_channels / out_channels:** How many "types of info" come in and go out. A grayscale image has 1 channel. An RGB image has 3. The output has one channel per kernel/filter.- **kernel_size:** The size of the sliding window (e.g., 3 means 3x3)- **stride:** How far the kernel slides each step (default 1 = move by 1 pixel)- **padding:** Extra zeros added around the image border to control output size

In [ ]:
class Conv2d(nn.Module):    def __init__(        self,        in_channels: int,     # Number of input channels (1 for grayscale, 3 for RGB)        out_channels: int,    # Number of output channels (= number of different filters)        kernel_size: int,     # Size of the sliding window (3 means 3x3)        stride: int = 1,      # How far to slide each step        padding: int = 0,     # How many zeros to pad around the border    ):        super().__init__()        # Store all settings as attributes        self.in_channels = in_channels        self.out_channels = out_channels        self.kernel_size = kernel_size        self.stride = stride        self.padding = padding                # ---- WEIGHT INITIALIZATION ----        # The weight (kernel) shape is:        #   (out_channels, in_channels, kernel_size, kernel_size)        #        # For Conv2d(in_channels=3, out_channels=16, kernel_size=3):        #   weight shape = (16, 3, 3, 3)        #   That's 16 different 3x3x3 filters        #   Each filter looks at all 3 input channels simultaneously        #        # Kaiming initialization (same idea as Linear):        fan_in = in_channels * kernel_size * kernel_size  # total inputs per output element        sf = 1 / (fan_in ** 0.5)                          # scaling factor        weight = sf * (2 * t.rand(out_channels, in_channels, kernel_size, kernel_size) - 1)        self.weight = nn.Parameter(weight)        def forward(self, x: Tensor) -> Tensor:        return t.nn.functional.conv2d(x, self.weight, stride=self.stride, padding=self.padding)        # We use PyTorch's built-in conv2d FUNCTION here        # (The bonus section at the end has you build this from scratch!)        #        # x shape: (batch, in_channels, height, width)        # output shape: (batch, out_channels, new_height, new_width)        #        # new_height = (height + 2*padding - kernel_size) / stride + 1        # e.g., (28 + 0 - 3) / 1 + 1 = 26 (image shrinks slightly)        def extra_repr(self) -> str:        keys = ["in_channels", "out_channels", "kernel_size", "stride", "padding"]        return ", ".join([f"{key}={getattr(self, key)}" for key in keys])        # getattr(self, key) is like self.key, but key is a string variable        # This is a compact way to print all the settings

## MaxPool2d - Shrinking the ImageAfter a convolution extracts features, **max pooling** shrinks the image by keeping only the strongest signal in each region.**Analogy:** Imagine tiling your image with non-overlapping 2x2 squares. In each square, you keep only the LARGEST value and throw away the other 3. This halves both dimensions: a 28x28 image becomes 14x14.**Why?** 1. Reduces computation (fewer pixels to process in later layers)2. Makes the network more robust to small shifts in the input (if a feature moves 1 pixel, the max in that region probably doesn't change)

In [ ]:
class MaxPool2d(nn.Module):    def __init__(self, kernel_size: int, stride: int | None = None, padding: int = 1):        # "int | None" means "either an int OR None" (Python 3.10+ syntax)        super().__init__()        self.kernel_size = kernel_size        self.stride = stride     # If None, defaults to kernel_size (non-overlapping tiles)        self.padding = padding        def forward(self, x: Tensor) -> Tensor:        return F.max_pool2d(x, kernel_size=self.kernel_size, stride=self.stride, padding=self.padding)        # For each kernel_size x kernel_size region, keeps only the maximum value        # No learnable parameters! This is a fixed operation.        def extra_repr(self) -> str:        return ", ".join([f"{key}={getattr(self, key)}" for key in ["kernel_size", "stride", "padding"]])

---# Part 5: ResNets (Residual Networks)## Sequential - Chaining Modules TogetherBefore building ResNet, we need a `Sequential` container. This is like a pipeline: data goes in one end, passes through each module in order, and comes out the other end.Instead of writing:```pythonx = module1(x)x = module2(x)x = module3(x)```You can write:```pythonpipeline = Sequential(module1, module2, module3)x = pipeline(x)```

In [ ]:
class Sequential(nn.Module):    _modules: dict[str, nn.Module]    # _modules is a special attribute that nn.Module uses internally    # It's a dictionary (lookup table) mapping names to modules    # dict[str, nn.Module] means: keys are strings, values are modules        def __init__(self, *modules: nn.Module):        # *modules means "accept ANY number of arguments"        # Sequential(a, b, c) -> modules = (a, b, c) as a tuple                super().__init__()        for index, mod in enumerate(modules):            self._modules[str(index)] = mod        # enumerate() gives you both the index AND the item:        #   enumerate([a, b, c]) -> (0, a), (1, b), (2, c)        # We store each module with its index as a string key:        #   {"0": module_a, "1": module_b, "2": module_c}        # Storing in _modules lets PyTorch auto-discover all sub-modules        def __getitem__(self, index: int) -> nn.Module:        # __getitem__ lets you use square bracket notation:        #   seq[0] calls seq.__getitem__(0)        index %= len(self._modules)   # Handle negative indices: -1 = last        return self._modules[str(index)]        def __setitem__(self, index: int, module: nn.Module) -> None:        # Lets you replace a module: seq[0] = new_module        index %= len(self._modules)        self._modules[str(index)] = module        def forward(self, x: Tensor) -> Tensor:        for mod in self._modules.values():            x = mod(x)        # Pass data through each module in order        # .values() gives just the modules (not the string keys)        # Each module transforms x, and the result becomes input to the next        return x

## BatchNorm2d - Keeping Numbers Well-Behaved**The problem:** As data flows through many layers, the numbers can drift — getting very large, very small, or very uneven. This makes training unstable and slow.**The solution:** After each convolutional layer, **normalize** the data. BatchNorm forces each channel's values to have mean ≈ 0 and variance ≈ 1, then lets the network learn a small scale and shift on top of that.**Analogy:** Imagine each layer is a person in a game of telephone. Without BatchNorm, the message gets more distorted with each person. BatchNorm is like having a "fact-checker" after each person who says "wait, let me normalize the message to make sure it's still clear" before passing it on.### Buffers vs Parameters- **Parameters** (`self.weight`, `self.bias`): learned via gradient descent (backpropagation)- **Buffers** (`running_mean`, `running_var`): updated by a formula, NOT by gradients. They track long-term statistics of the data passing through.### Train vs Eval Mode- **Training:** Uses the current batch's mean/variance to normalize. Also updates the running statistics.- **Evaluation:** Uses the stored running statistics (accumulated over all training batches). This ensures consistent behavior when processing single images.

In [ ]:
class BatchNorm2d(nn.Module):    running_mean: Float[Tensor, " num_features"]     # Type hints (documentation only)    running_var: Float[Tensor, " num_features"]    num_batches_tracked: Int[Tensor, ""]              # Scalar (0-dimensional) tensor        def __init__(self, num_features: int, eps=1e-05, momentum=0.1):        # num_features: number of channels (e.g., 64 for a layer with 64 channels)        # eps: tiny number added to denominator to prevent division by zero        # momentum: how fast running stats update (0.1 = update by 10% each batch)                super().__init__()        self.num_features = num_features        self.eps = eps        self.momentum = momentum                # LEARNABLE parameters (updated by gradient descent):        self.weight = nn.Parameter(t.ones(num_features))    # Scale (starts at 1)        self.bias = nn.Parameter(t.zeros(num_features))      # Shift (starts at 0)        # Initially these do nothing: multiply by 1, add 0        # Training will adjust them to optimal values                # BUFFERS (updated by formula, not gradient descent):        self.register_buffer("running_mean", t.zeros(num_features))        self.register_buffer("running_var", t.ones(num_features))        self.register_buffer("num_batches_tracked", t.tensor(0))        # register_buffer tells PyTorch: "save this tensor with the model,        # move it to GPU when model moves, BUT don't compute gradients for it"        #        # running_mean starts at 0, running_var starts at 1        # These get updated each training batch to track overall data statistics        def forward(self, x: Tensor) -> Tensor:        # The actual normalization:        # For each channel c:        #   output[c] = weight[c] * (input[c] - mean[c]) / sqrt(var[c] + eps) + bias[c]        #        # During training: mean and var are computed from the current batch        # During eval: mean and var are the running averages from training        raise NotImplementedError()        # This is the exercise - you fill in the formula!

## AveragePool - Collapsing Spatial DimensionsAverage pooling takes the average across the entire height and width of each channel. A tensor of shape `(batch, channels, height, width)` becomes `(batch, channels)`.**Analogy:** If each channel is a "feature detector" that produces a heatmap, average pooling says "how active was this feature detector overall?" by averaging the whole heatmap into a single number.

In [ ]:
class AveragePool(nn.Module):    def forward(self, x: Tensor) -> Tensor:        # x shape: (batch, channels, height, width)        # Output shape: (batch, channels)        return x.mean(dim=(-2, -1))        # .mean(dim=(-2, -1)) averages over the last two dimensions (height, width)        # dim=-2 is height, dim=-1 is width        # Negative indices count from the end: -1 = last, -2 = second-to-last        #         # For a tensor of shape (64, 512, 7, 7):        #   averages each 7x7 grid -> produces (64, 512)

## ResidualBlock - The Key Innovation of ResNetsThe **residual block** is what makes ResNets special. Instead of just transforming the input, it adds the input BACK to the output:```output = F(input) + input```Where `F` is a stack of Conv -> BatchNorm -> ReLU -> Conv -> BatchNorm.**Why is this brilliant?** In very deep networks (50+ layers), gradients can vanish — the learning signal fades to near-zero as it passes back through many layers. The skip connection (adding input to output) provides a "highway" for gradients to flow directly backward. Even if the layers learn nothing useful, the network can still pass information through via the skip connection.**Analogy:** Imagine a chain of editors reviewing a document. Without skip connections, each editor only sees the previous editor's version — after 50 editors, the original meaning might be lost. With skip connections, each editor also gets a copy of what came IN, so they can make incremental improvements rather than starting from scratch.

In [ ]:
class ResidualBlock(nn.Module):    def __init__(self, in_feats: int, out_feats: int, first_stride=1):        # in_feats: input channels        # out_feats: output channels        # first_stride: if > 1, this block reduces spatial size (downsampling)                super().__init__()        is_shape_preserving = (first_stride == 1) and (in_feats == out_feats)        # If stride=1 AND channels don't change, the input and output have        # the same shape -> we can add them directly (identity skip connection)        # If not, we need a 1x1 conv to match shapes (projection skip connection)                # LEFT BRANCH (the "transformation" path):        # Conv -> BatchNorm -> ReLU -> Conv -> BatchNorm        self.left = Sequential(            Conv2d(in_feats, out_feats, kernel_size=3, stride=first_stride, padding=1),            BatchNorm2d(out_feats),            ReLU(),            Conv2d(out_feats, out_feats, kernel_size=3, stride=1, padding=1),            BatchNorm2d(out_feats),        )                # RIGHT BRANCH (the "skip connection"):        if is_shape_preserving:            self.right = nn.Identity()   # Do nothing - just pass input through        else:            self.right = Sequential(                Conv2d(in_feats, out_feats, kernel_size=1, stride=first_stride),                BatchNorm2d(out_feats),            )            # 1x1 conv: changes number of channels without spatial processing            # stride matches left branch so spatial dimensions match too        def forward(self, x: Tensor) -> Tensor:        return self.left(x) + self.right(x)   # THE RESIDUAL CONNECTION!        # Run input through BOTH branches, then ADD the results        # This is the core idea that makes deep networks trainable

## BlockGroup and ResNet34 - Assembling the Full ArchitectureA **BlockGroup** is just several ResidualBlocks stacked together. ResNet34 has 4 groups with [3, 4, 6, 3] blocks respectively (3+4+6+3 = 16 blocks, each with 2 conv layers = 32 conv layers + initial conv + final linear = ~34 layers -> "ResNet34").The full ResNet34 architecture:```Input image (3, 224, 224)    |Conv2d(3->64, 7x7, stride=2) + BatchNorm + ReLU    -> (64, 112, 112)MaxPool(3x3, stride=2)                               -> (64, 56, 56)    |BlockGroup(3 blocks, 64->64)                         -> (64, 56, 56)BlockGroup(4 blocks, 64->128, stride=2)              -> (128, 28, 28)BlockGroup(6 blocks, 128->256, stride=2)             -> (256, 14, 14)BlockGroup(3 blocks, 256->512, stride=2)             -> (512, 7, 7)    |AveragePool                                          -> (512)Linear(512 -> 1000)                                  -> (1000 class scores)```

In [ ]:
class BlockGroup(nn.Module):    def __init__(self, n_blocks: int, in_feats: int, out_feats: int, first_stride=1):        super().__init__()        # First block might change dimensions (stride > 1 or different channel count)        # Remaining blocks preserve dimensions        blocks = [ResidualBlock(in_feats, out_feats, first_stride)]  # First block        for _ in range(1, n_blocks):            blocks.append(ResidualBlock(out_feats, out_feats, 1))     # Remaining blocks            # _ is convention for "I don't need this loop variable"            # After the first block, input = output = out_feats, stride = 1        self.blocks = Sequential(*blocks)        # *blocks "unpacks" the list: Sequential(*[a, b, c]) = Sequential(a, b, c)        def forward(self, x: Tensor) -> Tensor:        return self.blocks(x)

In [ ]:
class ResNet34(nn.Module):    def __init__(        self,        n_blocks_per_group=[3, 4, 6, 3],        out_features_per_group=[64, 128, 256, 512],        first_strides_per_group=[1, 2, 2, 2],        n_classes=1000,    # ImageNet has 1000 categories    ):        super().__init__()                # ---- INITIAL LAYERS (before the residual blocks) ----        self.in_layers = Sequential(            Conv2d(3, 64, kernel_size=7, stride=2, padding=3),  # 3 RGB channels -> 64            BatchNorm2d(64),            ReLU(),            MaxPool2d(kernel_size=3, stride=2, padding=1),        )                # ---- RESIDUAL BLOCK GROUPS ----        # zip() pairs up elements from multiple lists:        #   zip([3,4,6,3], [64,128,256,512], [1,2,2,2])        #   -> (3,64,1), (4,128,2), (6,256,2), (3,512,2)        in_features_per_group = [64] + out_features_per_group[:-1]        # [64] + [64, 128, 256] = [64, 64, 128, 256]        # Each group's input = previous group's output                self.residual_layers = Sequential(            *[                BlockGroup(n, in_f, out_f, stride)                for n, in_f, out_f, stride in zip(                    n_blocks_per_group,                    in_features_per_group,                    out_features_per_group,                    first_strides_per_group,                )            ]        )        # This LIST COMPREHENSION creates 4 BlockGroups and unpacks them into Sequential                # ---- FINAL LAYERS ----        self.out_layers = Sequential(            AveragePool(),            # (batch, 512, 7, 7) -> (batch, 512)            Flatten(),                # Already flat, but just in case            Linear(512, n_classes),   # 512 features -> 1000 class scores        )        def forward(self, x: Tensor) -> Tensor:        x = self.in_layers(x)        x = self.residual_layers(x)        x = self.out_layers(x)        return x

## Copying Pretrained WeightsInstead of training ResNet34 from scratch (which takes days on ImageNet), we copy weights from a model that PyTorch already trained. This is called using a **pretrained model**.**Analogy:** Instead of teaching a student from kindergarten, you hire someone who already graduated and put them to work immediately.

In [ ]:
def copy_weights(my_resnet: ResNet34, pretrained_resnet: models.resnet.ResNet) -> ResNet34:    # Get the "state dictionary" from both models    # A state dict is a mapping: parameter_name -> tensor_of_values    # e.g., {"linear1.weight": tensor([...]), "linear1.bias": tensor([...]), ...}        mydict = my_resnet.state_dict()    pretraineddict = pretrained_resnet.state_dict()        assert len(mydict) == len(pretraineddict), "Mismatching state dictionaries."    # Sanity check: both models should have the same number of parameters        # Create a new dict that maps OUR parameter names to THEIR values    state_dict_to_load = {        mykey: pretrainedvalue        for (mykey, myvalue), (pretrainedkey, pretrainedvalue)         in zip(mydict.items(), pretraineddict.items())    }    # .items() gives (key, value) pairs    # zip pairs them up: our first param with their first param, etc.    # We keep OUR names but use THEIR values        my_resnet.load_state_dict(state_dict_to_load)    # load_state_dict copies all the tensor values into our model        return my_resnet# Load PyTorch's pretrained ResNet34 (trained on ImageNet for days on powerful GPUs)pretrained_resnet = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1).to(device)# weights=...IMAGENET1K_V1 means "use the weights trained on ImageNet"# Without this argument, you'd get random weights# Copy those weights into our hand-built ResNet34my_resnet = copy_weights(my_resnet, pretrained_resnet).to(device)

## Loading Test Images and Making Predictions

In [ ]:
# ---- LOADING IMAGES ----IMAGE_FILENAMES = [    "chimpanzee.jpg", "golden_retriever.jpg", "platypus.jpg",    "frogs.jpg", "fireworks.jpg", "astronaut.jpg",    "iguana.jpg", "volcano.jpg", "goofy.jpg", "dragonfly.jpg",]IMAGE_FOLDER = section_dir / "resnet_inputs"images = [Image.open(IMAGE_FOLDER / filename) for filename in IMAGE_FILENAMES]# List comprehension: for each filename, open it as a PIL Image# Image.open() reads a .jpg file into a PIL Image object# This is like having a stack of photos on your desk# ---- PREPARING IMAGES FOR THE MODEL ----IMAGE_SIZE = 224            # ResNet expects 224x224 imagesIMAGENET_MEAN = [0.485, 0.456, 0.406]  # Average R, G, B values in ImageNetIMAGENET_STD = [0.229, 0.224, 0.225]    # Std deviation of R, G, B in ImageNetIMAGENET_TRANSFORM = transforms.Compose([    transforms.ToTensor(),                              # PIL Image -> Tensor, scale to [0,1]    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),        # Resize to 224x224    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),  # Normalize per channel])# Same idea as MNIST_TRANSFORM, but for RGB images with ImageNet statisticsprepared_images = t.stack([IMAGENET_TRANSFORM(img) for img in images], dim=0).to(device)# 1. Apply transform to each image -> list of tensors, each (3, 224, 224)# 2. t.stack([...], dim=0) stacks them into one tensor: (10, 3, 224, 224)#    10 images, 3 color channels, 224x224 pixels# 3. .to(device) moves to GPU

## The Predict Function

In [ ]:
@t.inference_mode()# This decorator is like wrapping the ENTIRE function in "with t.inference_mode():"# No gradients will be tracked for anything inside this functiondef predict(    model: nn.Module, images: Float[Tensor, "batch rgb h w"]) -> tuple[Float[Tensor, " batch"], Int[Tensor, " batch"]]:    # Returns: (probabilities, predicted_class_indices)    # Float[Tensor, "batch rgb h w"] is a type hint describing the tensor shape        model.eval()    # Switch model to EVALUATION MODE    # This affects BatchNorm (uses running stats instead of batch stats)    # and Dropout (disabled during eval) if present        logits = model(images)    # Shape: (batch, 1000) - one score per ImageNet class        probs = logits.softmax(dim=-1)    # Softmax converts raw scores to probabilities (all positive, sum to 1)    # dim=-1 means "apply along the last dimension" (the 1000 classes)        top_probs, top_indices = probs.max(dim=-1)    # .max() returns BOTH the maximum value AND its index    # top_probs: highest probability for each image    # top_indices: which class has that probability        return top_probs, top_indices# ---- RUNNING PREDICTIONS ----with open(section_dir / "imagenet_labels.json") as f:    imagenet_labels = list(json.load(f).values())# Load the mapping from class index to human-readable name# e.g., index 281 -> "tabby cat"my_probs, my_predictions = predict(my_resnet, prepared_images)# Run our hand-built model on the 10 test images

## Aside: Hooks (Debugging Neural Networks)**Hooks** are functions that get called automatically at certain points during the forward pass. They let you "peek inside" the network without modifying its code.**Analogy:** Imagine putting a camera at each station on an assembly line. You're not changing how the line works — you're just observing what happens at each point. If something goes wrong (a NaN appears), the camera catches it.

In [ ]:
class NanModule(nn.Module):    # A deliberately broken module that always outputs NaN (Not a Number)    # Used to demonstrate how hooks can catch errors    def forward(self, x):        return t.full_like(x, float("nan"))        # t.full_like(x, value) creates a tensor the same shape as x, filled with value        # float("nan") is the special "Not a Number" valuedef hook_check_for_nan_output(module: nn.Module, input: tuple[Tensor], output: Tensor) -> None:    # A hook function. PyTorch calls this AFTER each module's forward pass.    # module: which layer just ran    # input: what went IN to the layer (as a tuple)    # output: what came OUT    if t.isnan(output).any():        # t.isnan() checks each element: is it NaN?        # .any() returns True if ANY element is NaN        raise ValueError(f"NaN output from {module}")        # Raise an error so we know exactly WHICH module produced NaNdef add_hook(module: nn.Module) -> None:    module.register_forward_hook(hook_check_for_nan_output)    # register_forward_hook: "After this module runs forward(), also run this hook function"# ---- DEMO ----model = nn.Sequential(nn.Identity(), NanModule(), nn.Identity())# nn.Identity() does nothing (passes input through unchanged)# So data flows: Identity -> NanModule (breaks!) -> Identitymodel = model.apply(add_hook)# .apply(fn) calls fn on the model AND every sub-module recursively# So all 3 modules get the NaN-checking hooktry:    input = t.randn(3)     # Random input tensor    output = model(input)   # This will trigger the hook at NanModuleexcept ValueError as e:    print(e)               # "NaN output from NanModule()"# try/except catches the error so the notebook doesn't crash# The error tells us exactly which module caused the problem

---# Part 6: Bonus - Feature Extraction## What is Feature Extraction?**The idea:** Take a model pre-trained on a huge dataset (ImageNet, 1000 classes) and repurpose it for a different task (CIFAR-10, 10 classes). Instead of training all 21 million parameters from scratch, you:1. **Freeze** all layers except the last one (lock the weights so they don't change)2. **Replace** the last layer with a new one sized for your task3. **Train** only the new last layer**Why does this work?** The early layers of a CNN learn universal features: edges, textures, shapes. These are useful for ANY image task. Only the final layer is task-specific ("this combination of features = golden retriever"). So you keep the universal feature detectors and just retrain the "decision maker."**Analogy:** Imagine hiring an experienced detective and assigning them to a new city. Their investigation skills (feature extraction) transfer perfectly — you just need to teach them the local geography (the new classification layer).### `requires_grad_(False)` - Freezing Weights

In [ ]:
layer0, layer1 = nn.Linear(3, 4), nn.Linear(4, 5)# Create two simple linear layerslayer0.requires_grad_(False)# FREEZE layer0: its weights will NOT be updated during training# The trailing _ means "in-place" - it modifies layer0 directly# This sets param.requires_grad = False for ALL parameters in layer0## requires_grad is a flag on each tensor:#   True = "track gradients for this tensor" (it's being trained)#   False = "don't bother" (it's frozen)x = t.randn(3)                 # Random inputout = layer1(layer0(x)).sum()   # Forward pass through both layers, sum to scalarout.backward()                  # Compute gradientsassert layer0.weight.grad is None       # layer0 was frozen -> no gradients computed!assert layer1.weight.grad is not None   # layer1 is trainable -> gradients exist# This is the mechanism behind feature extraction:# Freeze the pretrained layers, only train the new ones

## Preparing ResNet for Feature Extraction

In [ ]:
def get_resnet_for_feature_extraction(n_classes: int) -> ResNet34:    # 1. Load pretrained ResNet34    my_resnet = ResNet34()    pretrained = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)    my_resnet = copy_weights(my_resnet, pretrained)        # 2. Freeze ALL weights    my_resnet.requires_grad_(False)        # 3. Replace the final linear layer with a NEW one (for n_classes instead of 1000)    my_resnet.out_layers[-1] = Linear(512, n_classes)    # out_layers[-1] is the last module in out_layers (the Linear layer)    # The new Linear layer has requires_grad=True by default (it's trainable!)    # So we've frozen everything EXCEPT the brand new final layer        return my_resnet

## CIFAR-10 Dataset and Training for Feature Extraction

In [ ]:
def get_cifar() -> tuple[datasets.CIFAR10, datasets.CIFAR10]:    # Same pattern as get_mnist, but for CIFAR-10    # CIFAR-10: 60,000 32x32 color images in 10 classes    # (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck)    cifar_trainset = datasets.CIFAR10(        exercises_dir / "data", train=True, download=True, transform=IMAGENET_TRANSFORM    )    cifar_testset = datasets.CIFAR10(        exercises_dir / "data", train=False, download=True, transform=IMAGENET_TRANSFORM    )    # Note: we use IMAGENET_TRANSFORM (not MNIST_TRANSFORM)    # because ResNet expects 224x224 RGB images normalized with ImageNet stats    # The transform resizes CIFAR's tiny 32x32 images up to 224x224    return cifar_trainset, cifar_testset@dataclassclass ResNetTrainingArgs:    batch_size: int = 64    epochs: int = 5        # More epochs since we're only training one layer    learning_rate: float = 1e-3    n_classes: int = 10    # CIFAR-10 has 10 classes

In [ ]:
def get_cifar_subset(trainset_size: int = 10_000, testset_size: int = 1_000) -> tuple[Subset, Subset]:    cifar_trainset, cifar_testset = get_cifar()    return Subset(cifar_trainset, range(trainset_size)), Subset(cifar_testset, range(testset_size))def train(args: ResNetTrainingArgs) -> tuple[list[float], list[float], ResNet34]:    # Very similar to the SimpleMLP training loop, but with:    # 1. A ResNet model (frozen except final layer) instead of SimpleMLP    # 2. CIFAR-10 data instead of MNIST    # 3. The same training + validation loop pattern        model = get_resnet_for_feature_extraction(args.n_classes).to(device)        cifar_trainset, cifar_testset = get_cifar_subset()    train_loader = DataLoader(cifar_trainset, batch_size=args.batch_size, shuffle=True)    test_loader = DataLoader(cifar_testset, batch_size=args.batch_size, shuffle=False)        # Only optimize the parameters that require gradients (the unfrozen ones)    optimizer = t.optim.Adam(        filter(lambda p: p.requires_grad, model.parameters()),        lr=args.learning_rate,    )    # filter(fn, iterable) keeps only items where fn returns True    # lambda p: p.requires_grad is an anonymous function that checks the flag    # So we ONLY give the optimizer the final layer's weights        loss_list = []    accuracy_list = []        for epoch in range(args.epochs):        # Training loop (same pattern as before)        model.train()    # Set to training mode (affects BatchNorm)        pbar = tqdm(train_loader)        for imgs, labels in pbar:            imgs, labels = imgs.to(device), labels.to(device)            logits = model(imgs)            loss = F.cross_entropy(logits, labels)            loss.backward()            optimizer.step()            optimizer.zero_grad()            loss_list.append(loss.item())            pbar.set_postfix(epoch=f"{epoch+1}/{args.epochs}", loss=f"{loss:.3f}")                # Validation loop (same pattern as before)        model.eval()     # Set to eval mode (affects BatchNorm)        num_correct = 0        for imgs, labels in test_loader:            imgs, labels = imgs.to(device), labels.to(device)            with t.inference_mode():                logits = model(imgs)            predictions = t.argmax(logits, dim=1)            num_correct += (predictions == labels).sum().item()        accuracy = num_correct / len(cifar_testset)        accuracy_list.append(accuracy)        return loss_list, accuracy_list, model

---# Part 7: Bonus - Convolutions From Scratch## Understanding `as_strided` - How Tensors Actually Work in MemoryThis section peels back the curtain on how PyTorch stores tensors in memory. Understanding this isn't required for using PyTorch, but it builds deep intuition for how operations like convolution actually work under the hood.### How a tensor lives in memoryA 2D tensor like:```[[0,  1,  2,  3,  4], [5,  6,  7,  8,  9], [10, 11, 12, 13, 14], [15, 16, 17, 18, 19]]```is stored as a **flat 1D block** in memory: `[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]`To navigate this flat block as if it were 2D, PyTorch uses two pieces of information:- **Size** (shape): `(4, 5)` — 4 rows, 5 columns- **Stride**: `(5, 1)` — to move down one row, skip 5 elements. To move right one column, skip 1 element.**Analogy:** Imagine a parking lot with 20 numbered spaces in a single row. The "tensor" is a 4x5 grid drawn on paper showing which car is in which logical position. The "stride" is the instruction: "to go to the next row of my grid, skip 5 parking spaces."### `as_strided` lets you create VIEWS with custom stridesYou can create a new "view" of the same memory with different size and stride. This doesn't copy data — it just changes how you interpret the same numbers. This is incredibly powerful (and dangerous if misused).

In [ ]:
test_input = t.tensor(    [        [0, 1, 2, 3, 4],        [5, 6, 7, 8, 9],        [10, 11, 12, 13, 14],        [15, 16, 17, 18, 19],    ],    dtype=t.float,    # dtype=t.float specifies the data type: 32-bit floating point numbers    # Even though these look like integers, we store them as floats    # because neural networks work with floats)# This tensor has:# shape = (4, 5)   -> 4 rows, 5 columns# stride = (5, 1)  -> skip 5 to go down a row, skip 1 to go right a column## Memory layout: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]#                 ^              ^                 ^                     ^#                 row 0 start    row 1 start       row 2 start           row 3 start#                 (index 0)      (index 5)         (index 10)            (index 15)

## Stride ExercisesThese exercises ask you to fill in `size` and `stride` to produce specific outputs from `test_input` using `as_strided`. This builds intuition for how tensor views work.

In [ ]:
TestCase = namedtuple("TestCase", ["output", "size", "stride"])# namedtuple creates a simple class with named fields# TestCase(output=..., size=..., stride=...) creates an object where# you can access .output, .size, .stridetest_cases = [    # ---- EXAMPLE 1: First row ----    TestCase(        output=t.tensor([0, 1, 2, 3]),        size=(4,),       # We want 4 elements        stride=(1,),     # Each element is 1 step from the next in memory    ),    # Start at element 0, take 4 elements stepping by 1: [0, 1, 2, 3]        # ---- EXAMPLE 2: Every other element from first two rows ----    TestCase(        output=t.tensor([[0, 2], [5, 7]]),        size=(2, 2),     # 2 rows, 2 columns        stride=(5, 2),   # Skip 5 for next row, skip 2 for next column    ),    # Row 0: start at 0, step by 2: [0, 2]    # Row 1: start at 0+5=5, step by 2: [5, 7]        # ---- EXERCISES: you fill in size and stride ----        # Exercise 1: [0, 1, 2, 3, 4] (first row)    TestCase(        output=t.tensor([0, 1, 2, 3, 4]),        size=(5,),       # 5 elements in a row        stride=(1,),     # Step by 1 (consecutive elements)    ),        # Exercise 2: [0, 5, 10, 15] (first column)    TestCase(        output=t.tensor([0, 5, 10, 15]),        size=(4,),       # 4 elements        stride=(5,),     # Step by 5 (jump to next row's first element)    ),        # Exercise 3: [[0, 1, 2], [5, 6, 7]] (top-left 2x3 block)    TestCase(        output=t.tensor([[0, 1, 2], [5, 6, 7]]),        size=(2, 3),     # 2 rows, 3 columns        stride=(5, 1),   # Skip 5 for rows, skip 1 for columns    ),        # Exercise 4: [[0, 1, 2], [10, 11, 12]] (rows 0 and 2, first 3 cols)    TestCase(        output=t.tensor([[0, 1, 2], [10, 11, 12]]),        size=(2, 3),     # 2 rows, 3 columns        stride=(10, 1),  # Skip 10 for rows (skip a row!), skip 1 for columns    ),        # Exercise 5: [[0, 0, 0], [11, 11, 11]] (repeated elements!)    TestCase(        output=t.tensor([[0, 0, 0], [11, 11, 11]]),        size=(2, 3),     # 2 rows, 3 columns        stride=(11, 0),  # Skip 11 for rows, skip 0 for columns!        # stride=0 means "don't move" -> same element repeated!        # Row 0: element at index 0, repeated 3 times: [0, 0, 0]        # Row 1: element at index 11, repeated 3 times: [11, 11, 11]    ),        # Exercise 6: [0, 6, 12, 18] (diagonal)    TestCase(        output=t.tensor([0, 6, 12, 18]),        size=(4,),       # 4 elements        stride=(6,),     # Skip 6 each time (5 for next row + 1 for next column = diagonal)    ),]# ---- RUNNING THE TESTS ----for i, test_case in enumerate(test_cases):    # enumerate gives (index, item) pairs: (0, case0), (1, case1), ...        if (test_case.size is None) or (test_case.stride is None):        print(f"Test {i} failed: attempt missing.")    else:        actual = test_input.as_strided(size=test_case.size, stride=test_case.stride)        # as_strided creates a NEW VIEW of test_input with the given size and stride        # No data is copied! It's just a different way of reading the same memory                if (test_case.output != actual).any():            print(f"Test {i} failed")        else:            print(f"Test {i} passed!")

## Using `as_strided` for Linear AlgebraThese exercises use `as_strided` to implement common operations WITHOUT loops. The key insight: by cleverly choosing strides, you can make PyTorch "see" the data arranged in exactly the right way for element-wise operations, then just sum.

In [ ]:
def as_strided_trace(mat: Float[Tensor, "i j"]) -> Float[Tensor, ""]:    # TRACE = sum of diagonal elements    # For [[1,2],[3,4]], trace = 1 + 4 = 5    #    # Strategy: use as_strided to extract the diagonal, then sum it    # Diagonal elements are at positions (0,0), (1,1), (2,2), ...    # In memory: index 0, index (cols+1), index 2*(cols+1), ...    # So stride = cols + 1        rows, cols = mat.shape    return mat.as_strided(size=(min(rows, cols),), stride=(cols + 1,)).sum()    # size: we want min(rows, cols) diagonal elements    # stride: each diagonal element is (cols + 1) positions apart in memory    #   (skip one full row of 'cols' elements, plus 1 to move right)def as_strided_mv(mat: Float[Tensor, "i j"], vec: Float[Tensor, " j"]) -> Float[Tensor, " i"]:    # MATRIX-VECTOR MULTIPLY: each output element is the dot product of    # a row of the matrix with the vector    #    # Strategy:     # 1. Use as_strided to "broadcast" vec into a matrix the same size as mat    #    (repeat the vector for each row)    # 2. Multiply element-wise    # 3. Sum across columns        rows, cols = mat.shape    vec_expanded = vec.as_strided(size=(rows, cols), stride=(0, 1))    # stride=(0, 1) means: moving to next ROW doesn't advance (repeat!),    # moving to next COLUMN advances by 1 (normal)    # This "tiles" the vector vertically without copying memory        return (mat * vec_expanded).sum(dim=1)    # Element-wise multiply, then sum each rowdef as_strided_mm(matA: Float[Tensor, "i j"], matB: Float[Tensor, "j k"]) -> Float[Tensor, "i k"]:    # MATRIX-MATRIX MULTIPLY    # Each output[i, k] = sum over j of A[i,j] * B[j,k]    #    # Strategy: expand both matrices to 3D (i, j, k), multiply, sum over j        i, j = matA.shape    j2, k = matB.shape    assert j == j2        # Expand A to (i, j, k) by repeating along k dimension    A_expanded = matA.as_strided(size=(i, j, k), stride=(j, 1, 0))    # stride=(j, 1, 0): moving along i skips j elements (normal row stride),    # moving along j skips 1 (normal), moving along k skips 0 (repeat!)        # Expand B to (i, j, k) by repeating along i dimension    B_expanded = matB.as_strided(size=(i, j, k), stride=(0, k, 1))    # stride=(0, k, 1): moving along i skips 0 (repeat!),    # moving along j skips k (normal row stride), moving along k skips 1 (normal)        return (A_expanded * B_expanded).sum(dim=1)    # Multiply element-wise in 3D, then sum over the j dimension

## Building Convolutions From ScratchNow the payoff: using `as_strided` to implement convolutions without any loops. The trick is to create a "windowed view" of the input where each position shows the kernel-sized patch it would convolve with.

In [ ]:
def conv1d_minimal_simple(    x: Float[Tensor, " width"], weights: Float[Tensor, " kernel_width"]) -> Float[Tensor, " output_width"]:    # SIMPLEST CASE: 1D convolution, single input, single kernel    #     # x = [a, b, c, d, e]  (width=5)    # weights = [w0, w1, w2] (kernel_width=3)    # output[0] = a*w0 + b*w1 + c*w2    # output[1] = b*w0 + c*w1 + d*w2      # output[2] = c*w0 + d*w1 + e*w2    # output_width = width - kernel_width + 1 = 5 - 3 + 1 = 3        width = x.shape[0]    kw = weights.shape[0]    out_width = width - kw + 1        # Create a windowed view using as_strided    x_strided = x.as_strided(size=(out_width, kw), stride=(1, 1))    # size=(out_width, kw): a 2D view with out_width rows, kw columns    # stride=(1, 1): both moving right in a row AND moving to next row advance by 1    #    # This creates overlapping windows:    # Row 0: x[0], x[1], x[2]  = [a, b, c]    # Row 1: x[1], x[2], x[3]  = [b, c, d]    # Row 2: x[2], x[3], x[4]  = [c, d, e]        return (x_strided * weights).sum(dim=1)    # Multiply each window by the kernel, sum across the window    # This IS the convolution!

In [ ]:
def conv1d_minimal(    x: Float[Tensor, "batch in_channels width"],    weights: Float[Tensor, "out_channels in_channels kernel_width"],) -> Float[Tensor, "batch out_channels output_width"]:    # FULL 1D CONV: handles batches, multiple input/output channels        batch, in_ch, width = x.shape    out_ch, in_ch2, kw = weights.shape    assert in_ch == in_ch2    out_width = width - kw + 1        # Create windowed view: (batch, in_channels, output_width, kernel_width)    x_strided = x.as_strided(        size=(batch, in_ch, out_width, kw),        stride=(x.stride(0), x.stride(1), 1, 1),        # x.stride(0) = stride to next batch element (preserves batch structure)        # x.stride(1) = stride to next channel (preserves channel structure)        # 1 = stride for sliding window position        # 1 = stride within the window    )        # Einsum: contract over in_channels and kernel_width    return einops.einsum(        x_strided, weights,        "batch in_ch out_w kw, out_ch in_ch kw -> batch out_ch out_w"    )

In [ ]:
def conv2d_minimal(    x: Float[Tensor, "batch in_channels height width"],    weights: Float[Tensor, "out_channels in_channels kernel_height kernel_width"],) -> Float[Tensor, "batch out_channels out_height out_width"]:    # 2D CONV: same idea, but sliding a 2D window over height AND width        batch, in_ch, h, w = x.shape    out_ch, in_ch2, kh, kw = weights.shape    out_h = h - kh + 1    out_w = w - kw + 1        # Create windowed view    x_strided = x.as_strided(        size=(batch, in_ch, out_h, out_w, kh, kw),        stride=(x.stride(0), x.stride(1), x.stride(2), x.stride(3), x.stride(2), x.stride(3)),        # Last two strides match height/width strides: the kernel window        # indexes into the same spatial dimensions as the main position    )        return einops.einsum(        x_strided, weights,        "batch in_ch out_h out_w kh kw, out_ch in_ch kh kw -> batch out_ch out_h out_w"    )

## Padding and Full Convolutions**Padding** adds zeros around the border of the input before convolving. Without padding, convolutions shrink the image. With `padding=1` and `kernel_size=3`, the output is the same size as the input.**Stride > 1** means the kernel skips positions, producing a smaller output. Stride=2 halves the dimensions.

In [ ]:
def pad1d(    x: Float[Tensor, "batch in_channels width"], left: int, right: int, pad_value: float) -> Float[Tensor, "batch in_channels width_padded"]:    # Add pad_value to left and right sides of the width dimension        batch, channels, width = x.shape    # Create a tensor of the right size, filled with pad_value    output = t.full((batch, channels, left + width + right), pad_value)    # t.full(size, value) creates a tensor filled with that value        output[..., left : left + width] = x    # [...] means "all preceding dimensions" (batch, channels)    # left : left + width is a SLICE: positions from 'left' to 'left + width - 1'    # We copy the original data into the middle, leaving padding on the edges        return outputdef pad2d(    x: Float[Tensor, "batch in_channels height width"],    left: int, right: int, top: int, bottom: int, pad_value: float,) -> Float[Tensor, "batch in_channels height_padded width_padded"]:    # Same idea but padding height AND width        batch, channels, height, width = x.shape    output = t.full(        (batch, channels, top + height + bottom, left + width + right), pad_value    )    output[..., top : top + height, left : left + width] = x    return output

In [ ]:
def conv1d(    x: Float[Tensor, "batch in_channels width"],    weights: Float[Tensor, "out_channels in_channels kernel_width"],    stride: int = 1,    padding: int = 0,) -> Float[Tensor, "batch out_channels output_width"]:    # Full conv1d with stride and padding support        # 1. Apply padding    if padding > 0:        x = pad1d(x, left=padding, right=padding, pad_value=0.0)        # 2. Compute output dimensions    batch, in_ch, width = x.shape    out_ch, in_ch2, kw = weights.shape    out_width = (width - kw) // stride + 1        # 3. Create strided view (stride parameter affects the window step)    x_strided = x.as_strided(        size=(batch, in_ch, out_width, kw),        stride=(x.stride(0), x.stride(1), stride, 1),        #                                 ^^^^^^        # stride parameter: instead of stepping by 1 between windows,        # step by 'stride' - this is what makes stride work!    )        return einops.einsum(        x_strided, weights,        "batch in_ch out_w kw, out_ch in_ch kw -> batch out_ch out_w"    )

In [ ]:
# ---- UTILITY: force_pair ----IntOrPair = int | tuple[int, int]# "int | tuple[int, int]" means "either a single int OR a tuple of two ints"# This is useful because many PyTorch functions accept both:#   stride=2 means stride=(2,2) for both height and width#   stride=(1,2) means stride=1 for height, stride=2 for widthPair = tuple[int, int]def force_pair(v: IntOrPair) -> Pair:    # Converts a single int to a pair: 2 -> (2, 2)    # Passes through a pair unchanged: (1, 2) -> (1, 2)    if isinstance(v, tuple):        if len(v) != 2:            raise ValueError(v)        return (int(v[0]), int(v[1]))    elif isinstance(v, int):        return (v, v)   # Duplicate: 2 -> (2, 2)    raise ValueError(v)

In [ ]:
def conv2d(    x: Float[Tensor, "batch in_channels height width"],    weights: Float[Tensor, "out_channels in_channels kernel_height kernel_width"],    stride: IntOrPair = 1,    padding: IntOrPair = 0,) -> Float[Tensor, "batch out_channels out_height out_width"]:    # Full 2D convolution - combines everything above        stride_h, stride_w = force_pair(stride)    pad_h, pad_w = force_pair(padding)        # Apply 2D padding    if pad_h > 0 or pad_w > 0:        x = pad2d(x, left=pad_w, right=pad_w, top=pad_h, bottom=pad_h, pad_value=0.0)        batch, in_ch, h, w = x.shape    out_ch, in_ch2, kh, kw = weights.shape    out_h = (h - kh) // stride_h + 1    out_w = (w - kw) // stride_w + 1        x_strided = x.as_strided(        size=(batch, in_ch, out_h, out_w, kh, kw),        stride=(x.stride(0), x.stride(1), x.stride(2) * stride_h, x.stride(3) * stride_w,                x.stride(2), x.stride(3)),    )        return einops.einsum(        x_strided, weights,        "batch in_ch oh ow kh kw, out_ch in_ch kh kw -> batch out_ch oh ow"    )

## Max Pooling From Scratch

In [ ]:
def maxpool2d(    x: Float[Tensor, "batch in_channels height width"],    kernel_size: IntOrPair,    stride: IntOrPair | None = None,    padding: IntOrPair = 0,) -> Float[Tensor, "batch in_channels out_height out_width"]:    # Same windowed-view strategy, but take MAX instead of weighted sum        kh, kw = force_pair(kernel_size)    if stride is None:        stride = (kh, kw)   # Default: non-overlapping windows    stride_h, stride_w = force_pair(stride)    pad_h, pad_w = force_pair(padding)        if pad_h > 0 or pad_w > 0:        x = pad2d(x, pad_w, pad_w, pad_h, pad_h, pad_value=float("-inf"))        # Pad with -infinity so padded values are never the maximum!        batch, channels, h, w = x.shape    out_h = (h - kh) // stride_h + 1    out_w = (w - kw) // stride_w + 1        x_strided = x.as_strided(        size=(batch, channels, out_h, out_w, kh, kw),        stride=(x.stride(0), x.stride(1), x.stride(2) * stride_h, x.stride(3) * stride_w,                x.stride(2), x.stride(3)),    )        return x_strided.amax(dim=(-2, -1))    # .amax(dim=(-2, -1)) takes the maximum over the last two dims (kh, kw)    # For each window, we keep the single largest value

---# Summary: The Complete PictureCongratulations on making it through! Here's a map of everything you've seen:## The Building Blocks (Modules)| Module | What it does | Learnable? ||---|---|---|| `ReLU` | Zeros out negatives | No || `Linear` | Matrix multiply + bias | Yes (weight, bias) || `Flatten` | Reshapes multi-dim to flat | No || `Conv2d` | Sliding window feature detection | Yes (kernels) || `MaxPool2d` | Shrinks by keeping maximums | No || `BatchNorm2d` | Normalizes channel values | Yes (scale, shift) || `AveragePool` | Averages across spatial dims | No |## The Architectures| Architecture | Structure | Use case ||---|---|---|| `SimpleMLP` | Flatten -> Linear -> ReLU -> Linear | Simple classification || `ResNet34` | Conv blocks with skip connections | Image classification |## The Training Loop1. **Forward pass:** data flows through the model, producing predictions2. **Loss computation:** measure how wrong predictions are3. **Backward pass:** compute gradients (which direction to adjust weights)4. **Optimizer step:** actually adjust the weights5. **Repeat** for thousands of batches across multiple epochs## Key Python/PyTorch Concepts| Concept | What it means ||---|---|| `class X(Y)` | X inherits from Y (X IS a type of Y) || `self` | "this specific object" inside a method || `__init__` | Constructor - runs when object is created || `nn.Parameter` | A tensor that gets trained (updated by optimizer) || `forward()` | What happens when data passes through a module || `.to(device)` | Move tensor/model to GPU or CPU || `loss.backward()` | Compute gradients for all parameters || `optimizer.step()` | Update parameters using gradients || `t.inference_mode()` | Disable gradient tracking (for eval/test) || `as_strided` | Create a view of memory with custom navigation |